In [1]:
import duckdb
con = duckdb.connect("olist.duckdb")

In [2]:
con.execute("""
CREATE OR REPLACE TABLE funnel AS
WITH counts AS (
  SELECT 1 AS step, 'Created' AS stage, COUNT(*) AS orders FROM orders
  UNION ALL
  SELECT 2, 'Approved', COUNT(order_approved_at) FROM orders
  UNION ALL
  SELECT 3, 'Shipped (handed to carrier)', COUNT(order_delivered_carrier_date) FROM orders
  UNION ALL
  SELECT 4, 'Delivered to customer', COUNT(order_delivered_customer_date) FROM orders
)
SELECT step, stage, orders,
       ROUND(100.0 * orders / FIRST_VALUE(orders) OVER (ORDER BY step), 2) AS pct_of_created,
       ROUND(100.0 * orders / LAG(orders) OVER (ORDER BY step), 2) AS pct_of_previous
FROM counts
ORDER BY step
""")

con.execute("SELECT * FROM funnel ORDER BY step").df()

,step,stage,orders,pct_of_created,pct_of_previous
0,1,Created,99441,100.00,NaN
1,2,Approved,99281,99.84,99.84
2,3,Shipped (handed to carrier),97658,98.21,98.37
3,4,Delivered to customer,96476,97.02,98.79


In [3]:
con.execute("""
CREATE OR REPLACE TABLE order_status_summary AS
SELECT order_status,
       COUNT(*) AS orders,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_orders
FROM orders
GROUP BY 1
ORDER BY orders DESC
""")

con.execute("SELECT * FROM order_status_summary").df()

,order_status,orders,pct_of_orders
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


In [4]:
con.execute("COPY funnel TO 'dashboard/funnel.csv' (HEADER, DELIMITER ',')")
con.execute("COPY order_status_summary TO 'dashboard/order_status.csv' (HEADER, DELIMITER ',')")
print("saved")

saved


In [5]:
f = con.execute("SELECT * FROM funnel ORDER BY step").df()
s = con.execute("SELECT * FROM order_status_summary").df().set_index("order_status")

In [6]:
def n(status):
    return int(s.loc[status, "orders"]) if status in s.index else 0

def p(status):
    return float(s.loc[status, "pct_of_orders"]) if status in s.index else 0.0

total = int(f.iloc[0].orders)
drops = f.iloc[1:].copy()
worst = drops.loc[drops.pct_of_previous.idxmin()]
prev_stage = f.loc[f.step == worst.step - 1, "stage"].iloc[0]

In [7]:
print("=== COPY THIS INTO business_questions.md (Q2 answer) ===")
print(f"- Answer: Of {total:,} orders, {f.iloc[1].pct_of_created}% were approved, {f.iloc[2].pct_of_created}% were handed to the carrier and {f.iloc[3].pct_of_created}% reached the customer. "
      f"The largest step-to-step drop is from {prev_stage} to {worst.stage} ({100 - worst.pct_of_previous:.2f}% of orders lost at that step). "
      f"{n('canceled'):,} orders were cancelled ({p('canceled')}%) and {n('unavailable'):,} were marked unavailable ({p('unavailable')}%). "
      f"{p('delivered')}% of orders have the status delivered; the rest are cancelled, unavailable or still in progress when the data was collected.")
print()
print("=== COPY THIS INTO README.md (under ## Method) ===")
print("- The order funnel uses order timestamps (created, approved, shipped, delivered) because Olist has no web clickstream. It covers all orders in the dataset.")

=== COPY THIS INTO business_questions.md (Q2 answer) ===
- Answer: Of 99,441 orders, 99.84% were approved, 98.21% were handed to the carrier and 97.02% reached the customer. The largest step-to-step drop is from Approved to Shipped (handed to carrier) (1.63% of orders lost at that step). 625 orders were cancelled (0.63%) and 609 were marked unavailable (0.61%). 97.02% of orders have the status delivered; the rest are cancelled, unavailable or still in progress when the data was collected.

=== COPY THIS INTO README.md (under ## Method) ===
- The order funnel uses order timestamps (created, approved, shipped, delivered) because Olist has no web clickstream. It covers all orders in the dataset.
